# AF2 confidence–IoU alignment audit
Validation-only audit setelah box-score factorial. Kandidat, box, dan kelas tetap; hanya dihitung headroom oracle confidence ordering. Tidak ada training dan test tidak dibuka.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys
from pathlib import Path
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'
BRANCH='codex/af2-quality-alignment-audit'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
else: raise RuntimeError('Git clone gagal tiga kali.')
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src'))
os.chdir(REPO)
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
print('REPO:',REPO,'| BRANCH:',BRANCH)

In [ ]:
import tarfile, torch
ARCHIVE_REL='bundles/faruq-development-v3-grouped.tar'
D0FT_REL='experiments/faruq-v3-acmc-optimization-control-v1/D0FT_seed42/weights/best.pt'
AF2_REL='experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt'
FACTORIAL_REL='experiments/faruq-v3-af2-box-score-factorial-v1/af2_box_score_factorial.json'
REQUIRED=(ARCHIVE_REL,D0FT_REL,AF2_REL,FACTORIAL_REL)
PROJECT=resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE,D0FT,AF2,FACTORIAL=[require_project_artifact(PROJECT,path) for path in REQUIRED]
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA/'data.yaml').is_file(),DATA
assert not (DATA/'test').exists(),'Test tidak boleh tersedia.'
GROUPED=DATA/'faruq_grouped_summary.json'
OUTPUT_ROOT=PROJECT/'experiments/faruq-v3-af2-quality-alignment-v1'
OUTPUT=OUTPUT_ROOT/'af2_quality_alignment.json'
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('PROJECT:',PROJECT); print('OUTPUT:',OUTPUT)

In [ ]:
import json
LOG=OUTPUT_ROOT/'af2_quality_alignment_run.log'; LOG.parent.mkdir(parents=True,exist_ok=True)
device='0' if torch.cuda.is_available() else 'cpu'
command=[sys.executable,'-u','-m','coffee_detector.analysis.af2_quality_alignment',
 '--d0ft-checkpoint',str(D0FT),'--af2-checkpoint',str(AF2),
 '--data-root',str(DATA),'--grouped-summary',str(GROUPED),
 '--factorial-summary',str(FACTORIAL),'--output',str(OUTPUT),
 '--device',device,'--max-det','500','--confidence','0.001']
print('MENJALANKAN VALIDATION-ONLY QUALITY ALIGNMENT AUDIT')
with LOG.open('w',encoding='utf-8') as stream:
    process=subprocess.run(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:]))
if process.returncode: raise RuntimeError(f'Audit gagal: {process.returncode}; log={LOG}')

In [ ]:
import pandas as pd
result=json.loads(OUTPUT.read_text())
rows=[]
for model,row in result['results'].items():
    rows.append({'model':model,
      'native_macro':row['native']['macro_map50_95'],
      'oracle_macro':row['fixed_candidate_iou_oracle']['macro_map50_95'],
      'macro_headroom':row['oracle_minus_native']['macro_map50_95'],
      'bottom3_headroom':row['oracle_minus_native']['bottom3_class_map50_95'],
      'worst_headroom':row['oracle_minus_native']['worst_class_map50_95'],
      'spearman':row['alignment']['spearman_confidence_quality'],
      'continuous_ece':row['alignment']['continuous_ece'],
      'quality_brier':row['alignment']['quality_brier']})
display(pd.DataFrame(rows).style.format({
 'native_macro':'{:.2%}','oracle_macro':'{:.2%}','macro_headroom':'{:+.2%}',
 'bottom3_headroom':'{:+.2%}','worst_headroom':'{:+.2%}',
 'spearman':'{:.3f}','continuous_ece':'{:.3f}','quality_brier':'{:.3f}'}))
print('GATES:',result['gates'])
print('COMPARISON:',json.dumps(result['comparison'],indent=2))
print('DECISION:',result['decision'])
print('NEXT:',result['next'])
print('TRAINING:',result['training_executed'],'| TEST:',result['test_images_accessed'])
print('SUMMARY:',OUTPUT)
print('Kirim tabel, gates, comparison, decision, dan next. Jangan training atau membuka test.')